<a href="https://colab.research.google.com/github/ayeza527/tmdb_project/blob/main/TMDB_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from google.colab import userdata

API_KEY = userdata.get("tmdb_key")
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": API_KEY,
    "primary_release_date.gte": "2016-01-01",
    "primary_release_date.lte": "2016-12-31",
    "sort_by": "vote_count.desc",
    "page": 1,
}
r = requests.get(url, params=params)
data = r.json()

print("Status:", r.status_code)
print("Total pages:", data["total_pages"])
print("Total movies:", data["total_results"])
print("Top movie:", data["results"][0]["title"])

Status: 200
Total pages: 1001
Total movies: 20001
Top movie: Deadpool


This is a test code to test that api key is working and we can access the data, this code requests only 1 year wort of data and 1 page:

In [8]:
import os
SAVE_DIR = "/content/data"
os.makedirs(SAVE_DIR, exist_ok=True)
print(os.path.exists(SAVE_DIR))

True


In [14]:
import os
import time
import requests
import pandas as pd
from datetime import datetime

DIR = "/content/data"
os.makedirs(DIR, exist_ok=True)

url = "https://api.themoviedb.org/3/discover/movie"

start_time = datetime.now()
req_count = 0
all_ids = []

for year in range(2016, 2027):
    year_ids = []
    for page in range(1, 6):
        r = requests.get(url, {
            "api_key": API_KEY,
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            "sort_by": "vote_count.desc",
            "include_adult": "false",
            "page": page,
        })
        req_count += 1
        data = r.json()

        year_ids += [movie["id"] for movie in data["results"]]
        time.sleep(0.1)
        # we are doing this so alot of request do not at the same time and there is a gap in between to avoid error

    print(year, len(year_ids))
    all_ids += year_ids

all_ids = list(dict.fromkeys(all_ids))
print("Unique IDs:", len(all_ids))
print("Total requests: ", req_count)
pd.Series(all_ids, name="id").to_csv(f"{DIR}/movie_ids.csv", index=False)

2016 100
2017 100
2018 100
2019 100
2020 100
2021 100
2022 100
2023 100
2024 100
2025 100
2026 100
Unique IDs: 1100
Total requests:  55


I used the TMDB discover movies endpoints to collect movie IDs for years 2016 tp 2026. For each year, i requested 5 pages of result to get a total of 100 movies(20 movies per page). I , then removed duplicates by toring itn a dictionary then saved it as a list in movie_ids.csv. Discover filters by release date but can extract little informaton at a time, so i got ids first so i can use them to extract other information in the next step. I also retrieved one year at a time to get an even spread across years, sorted by vote_count descending becase well known films are more likely to have a real budget and revenue values.added a short pause between requests to stay within TMDB’s rate limits, and I counted every request for the collection log.

In [15]:
import json
import hashlib

raw_movies = []
failed_ids = []

for i, movie_id in enumerate(all_ids):
    details_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    params = {"api_key": API_KEY, "append_to_response": "credits"}

    for attempt in range(3):
        r = requests.get(details_url, params=params)
        request_count += 1
        if r.status_code == 200:
            break
        time.sleep(2)

    if r.status_code == 200:
        movie = r.json()
        movie["collected_at"] = datetime.now().isoformat(timespec="seconds")
        raw_movies.append(movie)
    else:
        failed_ids.append(movie_id)

    if (i + 1) % 100 == 0:
        print("Done:", i + 1, "of", len(all_ids))
    time.sleep(0.05)

end_time = datetime.now()

raw_path = f"{DIR}/tmdb_raw.json"
with open(raw_path, "w") as f:
    json.dump(raw_movies, f)

with open(raw_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("Movies collected:", len(raw_movies))
print("Failed IDs:", len(failed_ids))
print("Total requests so far:", request_count)
print("SHA-256:", sha256)

Done: 100 of 1100
Done: 200 of 1100
Done: 300 of 1100
Done: 400 of 1100
Done: 500 of 1100
Done: 600 of 1100
Done: 700 of 1100
Done: 800 of 1100
Done: 900 of 1100
Done: 1000 of 1100
Done: 1100 of 1100
Movies collected: 1100
Failed IDs: 0
Total requests so far: 1100
SHA-256: b2eefe568711c6dfd620750d77516a08ad88484eb93cb24e07b34b5fc0a4f63d


For each of the ~1,100 movie IDs collected in the previous step, I requested the full details from the TMDB Movie Details endpoint, using append_to_response=credits to receive cast and crew in the same reply. I added a collected_at timestamp to every movie, saved all the unmodified replies to tmdb_raw.json, and calculated a SHA-256 fingerprint of that file.
SHA256 is a method that takes any file and turns it into a 64-character code whichis a fingerprint that shows this data was collected byou team at a specific time.I also added a retry (up to 3 attempts) and a short pause between requests so temporary failures and rate limits wouldn't break the collection.

In [18]:
import os, json, hashlib
import pandas as pd

SAVE_DIR = "/content/data"
raw_path = f"{SAVE_DIR}/tmdb_raw.json"
print(os.listdir(SAVE_DIR))

with open(raw_path) as f:
    raw_movies = json.load(f)

with open(raw_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("Movies:", len(raw_movies))
print("SHA-256:", sha256)

['tmdb_raw.json', 'movie_ids.csv']
Movies: 1100
SHA-256: b2eefe568711c6dfd620750d77516a08ad88484eb93cb24e07b34b5fc0a4f63d


In [19]:
stamps = sorted(m["collected_at"] for m in raw_movies)
start_time, end_time = stamps[0], stamps[-1]
print(start_time, end_time)


2026-11-23T16:03:33 2026-11-23T16:06:31


After the Colab session disconnected, the variables stored in memory were lost, so I reloaded the saved raw file tmdb_raw.json into raw_movies and recalculated its SHA-256 fingerprint, instead of collecting the data again, which would have created new timestamps and a different fingerprint. The output shows that the folder contains tmdb_raw.json and movie_ids.csv, that the file holds 1,100 movies, and that the recalculated SHA-256 is identical to the one printed at the end of the collection step. This confirms that the raw data was not modified, so the collection log from before the disconnect is still valid.he disconnect also erased the start_time and end_time variables that the data collection log requires, so I recovered them by sorting the collected_at column of the raw data and taking the earliest and latest values.